# Notebook 45 · Point-in-time integrity, leakage audit, and frozen-signal sensitivity

**Development-only · final historical holdout closed · prospective ledger untouched · zero transaction costs.**

**Important:** The local yfinance-derived dataset cannot certify point-in-time universe membership, corporate-action correctness, or delisted security completeness. This notebook reports what can be tested internally, flags what requires external source verification, and deliberately avoids a new model search. The Ridge baseline is frozen at α=100 with the Notebook 41 training pipeline. No test outcome is used to select a new live strategy.

## Experiments 1–5 · Load development data, document provenance and immutable settings

In [ ]:
from pathlib import Path
import sys,time,json,hashlib
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert (ROOT/'src').is_dir(),f'Run from repository root or notebooks/, not {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.edge_robustness_41 import fit_and_score,run_portfolio_stress
from stock_ml_dynamic.point_in_time_audit_45 import *
from stock_ml_dynamic.edge_source_research import *
MASTER=ROOT/'artifacts/master_tables/1d'
master,latest,bundle,manifest=load_master(MASTER)
print('Master',master.shape,'latest',latest.shape)
print('Manifest:',manifest)
print('Frozen configuration: Ridge alpha=100, cap=60000, seed=42, folds=3, positions=(5,10,20)')

## Experiments 6–12 · Master integrity: keys, target arithmetic, adjusted prices, and missing values

In [ ]:
integrity=master_integrity(master,latest,manifest)
display(integrity)
assert integrity.set_index('check').loc['duplicate_keys','value']==0
assert integrity.set_index('check').loc['nonmonotonic_target_timestamps','value']==0
assert integrity.set_index('check').loc['target_identity_over_1e-8','value']==0
print('Arithmetic consistency passed; this does NOT verify vendor price accuracy.')

## Experiments 13–19 · Symbol coverage, date gaps, missing latest records, and delisting limitations

In [ ]:
inventory=source_inventory(master,latest)
display(inventory.describe(include='all').T)
display(inventory.sort_values('gaps_over_7_calendar_days',ascending=False).head(25))
display(inventory.loc[~inventory.in_latest].head(40))
print('WARNING: No point-in-time membership history is present. This cannot certify survivorship-free results.')
print('Calendar gaps include holidays and legitimate market closures; investigate before labeling bad data.')

## Experiments 20–26 · Extreme-return source audit queue (no automatic data correction)

In [ ]:
extremes=extreme_observations(master,threshold=.20)
display(extremes.head(40))
print('Rows >20%:',len(extremes),'rows >50%:',int((master.next_return.abs()>.5).sum()))
print('Each flagged event requires vendor/corporate-action confirmation; flags are NOT proof of error.')
if len(extremes):
    display(extremes.groupby('symbol').size().sort_values(ascending=False).head(25).rename('flagged_events'))

## Experiments 27–32 · Reconstruct frozen development-only Ridge scores and temporal boundaries

In [ ]:
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
print('Development',development.shape,'closed holdout cutoff',cutoff)
expanded,historical=add_historical_market_features(development)
expanded,accuracy_cols=add_accuracy_features(expanded)
expanded,features=add_volume_regime_features(expanded)
print("Frozen model feature extras:",len(features))
t0=time.perf_counter()
scored,coefficients,fold_audit=fit_and_score(expanded,features,cap=60000,seed=42)
chronology=chronology_audit(master,scored,fold_audit)
display(chronology);display(fold_audit)
assert chronology.purged.all(),'Leakage: training labels overlap validation start'
print('Fitting seconds',round(time.perf_counter()-t0,1))

## Experiments 33–38 · Score integrity and frozen baseline reconciliation

In [ ]:
score_checks=score_integrity(scored)
base=run_portfolio_stress(scored)
base5=base['metrics'].loc[(base['metrics']['model']=='ridge_all')&(base['metrics']['k']==5)].sort_values('fold')
display(score_checks);display(base5)
assert score_checks.duplicate_stock_dates.eq(0).all()
assert score_checks.missing_scores.eq(0).all()
print('Expected Notebook 41 five-position fold means: approximately 0.000876, 0.001992, 0.001562')
print('Observed:',base5.mean_daily.to_list())
print('Differences may indicate changed upstream data; investigate before comparing notebooks.')

## Experiments 39–45 · Pre-registered stress tests on FIXED stock selections

These stress tests **do not** propose deployable filters. They alter realized outcomes *after* frozen selection only to measure how sensitive historical results are to extreme returns. Winsorizing is symmetric; dropping observations changes the evaluation population and must not be interpreted as a feasible trading rule.

In [ ]:
sensitivity,sensitivity_daily=frozen_sensitivity(scored,ks=(5,10,20),thresholds=(.10,.20,.50))
display(sensitivity.loc[sensitivity.k.eq(5)].pivot(index='scenario',columns='fold',values='mean_daily').round(6))
fig,ax=plt.subplots(figsize=(10,4))
sensitivity.loc[sensitivity.k.eq(5)].pivot(index='scenario',columns='fold',values='mean_daily').plot(kind='bar',ax=ax)
ax.axhline(0,color='black',lw=.8);ax.set_ylabel('Mean daily gross return');plt.tight_layout();plt.show()

## Experiments 46–51 · Moving-block uncertainty around tail sensitivity

In [ ]:
inference=bootstrap_sensitivity(sensitivity_daily,reps=1200)
display(inference.loc[inference.k.eq(5)].sort_values(['scenario','fold']).round(6))
print('These intervals describe retrospective stress-test differences, not independent model alpha.')

## Experiments 52–57 · Symbol concentration and source-level investigation priorities

In [ ]:
leaders,selected= symbol_influence(scored,k=5,top_n=20)
display(leaders)
flagged=set(extremes.symbol)
leaders['has_master_extreme_20pct']=leaders.symbol.isin(flagged)
display(leaders)
for f,g in selected.groupby('fold'):
    total=float(g.next_return.sum())
    n=max(1,int(np.ceil(len(g)*.01)))
    top=float(g.next_return.nlargest(n).sum())
    print('Fold',f,'top 1% contribution to net:',top/total if total else np.nan)
print('High influence is NOT evidence of invalid data.')

## Experiments 58–63 · Feature timing and pipeline code audit (manual review required)

In [ ]:
import inspect
from stock_ml_dynamic import master_table,candle_table,historical_market,accuracy_features,volume_regime_research,walk_forward,economic_target_research
modules=[master_table,candle_table,historical_market,accuracy_features,volume_regime_research,walk_forward,economic_target_research]
source_rows=[]
for m in modules:
    source=inspect.getsource(m)
    source_rows.append(dict(module=m.__name__,uses_shift_minus_1=('shift(-1)' in source or 'shift(- 1)' in source),
                            uses_bfill=('bfill(' in source or 'backfill(' in source),
                            uses_centered_rolling=('center=True' in source),
                            uses_global_fit=('fit(' in source),source_lines=len(source.splitlines())))
source_audit=pd.DataFrame(source_rows)
display(source_audit)
print('These are REVIEW FLAGS, not proof of leakage. shift(-1) is expected for labels but not predictors.')
print('Global master preprocessors/clusters must be checked against every fold boundary; master fitting cutoff alone is insufficient proof of fold-local fitting.')

## Experiments 64–70 · Survivorship, corporate actions, tradability and point-in-time verification checklist

In [ ]:
external_checks=pd.DataFrame([
 ('Point-in-time index/universe membership','UNVERIFIED','Historical membership table with effective dates; include delisted stocks'),
 ('Adjusted-close corporate actions','UNVERIFIED','Cross-check flagged events against split/dividend history and independent vendor'),
 ('Delisted securities / bankruptcies','UNVERIFIED','Delisting-return and symbol mapping history'),
 ('Feature timestamp availability','PARTIALLY AUDITED','Audit each predictor at the market-close decision timestamp'),
 ('Label next-bar alignment','INTERNAL CHECK','Validate next timestamp and arithmetic; cross-check independent price source'),
 ('Vendor revision / backfill drift','UNVERIFIED','Versioned historical snapshots and ingestion timestamps'),
 ('Market holiday / missing bars','PARTIALLY AUDITED','Compare to official exchange calendar and suspension events'),
 ('Market impact / executable fills','OUT OF SCOPE','Transaction costs intentionally zero; execution assumptions not verified'),
 ('Multiple-testing / repeated research','UNRESOLVED','Freeze candidate and protocol before independent evaluation'),
],columns=['risk','status','required_evidence'])
display(external_checks)
print('No external source has been queried by this notebook.')

## Experiments 71–76 · Frozen research protocol for eventual independent validation

In [ ]:
protocol={
 'status':'RESEARCH ONLY — no final holdout opened',
 'universe':'current 220-stock source-derived universe; point-in-time membership unverified',
 'features':'Notebook 41 frozen features, training-only fit per fold',
 'model':'Ridge(alpha=100), target clipped at training-only 0.5%/99.5%',
 'selection':'Top 5 by model score per validation day; equal weight',
 'benchmark':'Date-matched factor controls from Notebook 44, plus equal-weight universe',
 'primary_metric':'Daily market-relative gross return; block-bootstrap confidence interval',
 'secondary_metrics':['profit factor','payoff ratio','drawdown','top 1% contribution','tail sensitivity'],
 'cost_assumption':'0, by user request',
 'acceptance':'No promotion without independent validation, source verification, and multiplicity-aware evidence',
 'holdout':'CLOSED; not accessed or used for decisions',
 'prospective_ledger':'UNTOUCHED',
}
print(json.dumps(protocol,indent=2))

## Experiments 77–82 · Export auditable results and final decision

In [ ]:
OUT=ROOT/'artifacts/point_in_time_audit/1d'
save_tables(OUT,master_integrity=integrity,source_inventory=inventory,extreme_observations=extremes,
            chronology=chronology,fold_audit=fold_audit,score_integrity=score_checks,
            baseline=base5,sensitivity=sensitivity,sensitivity_daily=sensitivity_daily,
            bootstrap_inference=inference,symbol_contributors=leaders,source_code_review=source_audit,
            external_verification=external_checks)
(OUT/'frozen_validation_protocol.json').write_text(json.dumps(protocol,indent=2))
print('Saved',OUT)
print('Decision: NO promotion. Verify point-in-time universe, adjusted prices, and feature timing before opening independent holdout.')